# Practical 1: Simulate and interpret diffuse fracture damage

**Learning objective:** Build a small public PhAST calculation, apply symmetric tension, and inspect displacement, damage and reaction. Compare two imposed separations using saved numerical results.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CEMS-Lab/autumn-school/blob/main/notebooks/study/classroom/01_simulate_fracture.ipynb)

[Download Practice Notebook](https://cems-lab.github.io/autumn-school/notebooks/study/classroom/01_simulate_fracture.ipynb) · [Download with Worked Solutions](https://cems-lab.github.io/autumn-school/notebooks/solutions/classroom/01_simulate_fracture.ipynb) · [Environment Setup](https://cems-lab.github.io/autumn-school/SETUP.md)

**Predict first:** Where will damage increase, and how should halving the separation affect the elastic energy?

Run the cells in order. The setup and figure-formatting cells can be expanded when you want to inspect them. The principal model, derivative and learning operations stay visible.

Prepared by Allamaprabhu Ani and Sathiskumar A. Ponnusami, CEMS-Lab, for the UKACM Autumn School 2026.

In [ ]:
#@title Set up the course environment { display-mode: "form" }
# Environment setup is measured separately from the classroom computation.
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys
import time

setup_started = time.perf_counter()
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("This PhAST snapshot supports Python 3.10–3.12. Select a compatible runtime; see Environment Setup.")

def locate_course_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "autumn-school", base / "teaching/ukacm_autumn_school_2026"):
            if (candidate / "notebooks/day2_helpers").is_dir() and (candidate / "vendor/PhAST/src").is_dir():
                return candidate.resolve()
    return None

COURSE_ROOT = locate_course_root()
if COURSE_ROOT is None and IN_COLAB:
    COURSE_ROOT = Path.cwd() / "autumn-school"
    if COURSE_ROOT.exists():
        raise RuntimeError("An incomplete autumn-school folder exists. Start a fresh runtime or select the complete course folder.")
    # A release tag or full commit can be supplied for a fixed course edition.
    course_ref = os.environ.get("PHAST_COURSE_REF", "main")
    subprocess.run(["git", "init", str(COURSE_ROOT)], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "remote", "add", "origin", "https://github.com/CEMS-Lab/autumn-school.git"], check=True)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "fetch", "--depth", "1", "origin", course_ref], check=True, timeout=180)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True, capture_output=True)
if COURSE_ROOT is None:
    raise FileNotFoundError("Open the notebook from the complete course folder, including notebooks/, configs/ and vendor/. See Environment Setup.")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(COURSE_ROOT / "vendor/PhAST"), "nbformat", "nbclient", "nbconvert"], check=True, timeout=600)

for package_dir in (COURSE_ROOT / "vendor/PhAST/src", COURSE_ROOT / "notebooks"):
    if str(package_dir) not in sys.path:
        sys.path.insert(0, str(package_dir))

revision = subprocess.run(["git", "-C", str(COURSE_ROOT), "rev-parse", "HEAD"], text=True, capture_output=True)
package_versions = {}
for package in ("torch", "numpy", "scipy", "matplotlib", "nbformat"):
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = "install using Environment Setup"
import matplotlib.pyplot as plt
import numpy as np
import torch
from io import BytesIO
from IPython.display import Image, display
from day2_helpers import assets_dir

torch.set_default_dtype(torch.float64)
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    "figure.titlesize": 13, "figure.dpi": 150, "axes.grid": True,
    "grid.alpha": 0.3, "grid.linestyle": "--", "lines.linewidth": 1.8,
})

def display_figure(figure, dpi=150, alt="Rendered teaching figure"):
    """Retain figures for reading in the book and in a fresh notebook session."""
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue(), alt=alt))

# Include dependency imports and plotting configuration in the setup duration.
setup_summary = {
    "environment": "Google Colab" if IN_COLAB else "local",
    "python": sys.version.split()[0],
    "course_revision": revision.stdout.strip() if revision.returncode == 0 else "downloaded archive",
    "versions": package_versions,
    "setup_seconds": round(time.perf_counter() - setup_started, 3),
}
print(setup_summary)

session_started = time.perf_counter()
torch.set_num_threads(1)

## Physical motivation and model

Pulling the top and bottom of a specimen apart stores elastic energy. A phase-field model balances that energy with the cost of a diffuse damaged region. We use a $4\times2$ rectangle with an initially damaged centreline of nominal length $0.8$.

This actual PhAST example uses quasistatic AT2, isotropic energy degradation, plane strain and assembled sparse-direct mechanics. Its loading produces diffuse damage evolution. All quantities use consistent dimensionless teaching scales.

The damage variable is $d=0$ in intact material and $d=1$ on the initial notch. The regularisation length $\ell=0.15$ sets the scale over which damage varies. The energy takes the AT2 form

$$\mathcal E(\mathbf u,d)=\int_\Omega g(d)\,\psi_0(\boldsymbol\varepsilon(\mathbf u))\,\mathrm d\Omega + \frac{G_c}{2}\int_\Omega\left(\frac{d^2}{\ell}+\ell|\nabla d|^2\right)\mathrm d\Omega.$$

Here $\mathbf u$ is displacement, $\boldsymbol\varepsilon$ is small strain, $\psi_0$ is elastic energy density, $g$ degrades stiffness, and $G_c$ is fracture energy per unit crack area. Prescribed boundary motion supplies the loading.

## Step-by-step calculation

### Choose the public case

The single configuration supplies the geometry, mesh, material and loading. Source verification selects the public PhAST version bundled with this course.

In [ ]:
import copy
import matplotlib.tri as mtri
from day2_helpers import import_public_phast, load_forward_config
from day2_helpers.forward_workflow import prepare_course_mesh, solver_configuration
from day2_helpers.forward_workflow import solve_prepared_case, save_results, load_results
from day2_helpers.forward_workflow import digest_file, notebook_source_hash
config = load_forward_config()
phast_info = import_public_phast(config["public_source"]["revision"])
torch.manual_seed(config["seed"])
torch.set_num_threads(min(4, os.cpu_count() or 1))

The geometry sketch shows the imposed vertical motion and the lateral restraints. Its plotting details are expandable.

In [ ]:
practical_dir = assets_dir() / "classroom" / "01"
practical_dir.mkdir(parents=True, exist_ok=True)
geometry, mesh_cfg = config["geometry"], config["mesh"]
width, height, notch_length = (geometry[key] for key in ("width", "height", "notch_length"))
print({"geometry": geometry, "mesh": mesh_cfg, "loading": config["loading"]})
def show_figure(fig, name, alt):
    fig.savefig(practical_dir / name, dpi=150, bbox_inches="tight")
    display_figure(fig, alt=alt)
    plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8), layout="constrained")
ax.plot([0, width, width, 0, 0], [0, 0, height, height, 0], color="#245a81")
ax.plot([0, notch_length], [height/2, height/2], color="#d46b27", lw=5, label="Initial damaged centreline")
for x_arrow in np.linspace(.5, width-.5, 5):
    ax.annotate("", xy=(x_arrow, height+.27), xytext=(x_arrow, height), arrowprops={"arrowstyle": "->", "color": "#245a81"})
    ax.annotate("", xy=(x_arrow, -.27), xytext=(x_arrow, 0), arrowprops={"arrowstyle": "->", "color": "#245a81"})
ax.text(width/2, height+.35, r"$u_y=+\delta/2$", ha="center")
ax.text(width/2, -.48, r"$u_y=-\delta/2$", ha="center")
ax.text(width+.15, height/2, r"$u_x=0$ on all exterior edges", rotation=90, va="center")
ax.set(xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal",
       xlim=(-.12, width+.4), ylim=(-.58, height+.58), title="Geometry and loading")
ax.legend(loc="upper left", bbox_to_anchor=(.015, .79))
show_figure(fig, "tiny_notched_tension_geometry.png", "Rectangle with an initially damaged centreline and outward symmetric vertical loading arrows; horizontal displacement is restrained on every exterior edge.")

### Create the mesh

The course helper `prepare_course_mesh` creates the rectangular three-node triangular mesh, saves and reopens its arrays, checks the round trip, and returns the mesh used below. Coordinates have shape `(nodes, 2)`; connectivity has shape `(triangles, 3)`. The named boundary sets identify exterior nodes. [Inspect the helper source](https://github.com/CEMS-Lab/autumn-school/blob/main/notebooks/day2_helpers/forward_workflow.py).

In [ ]:
prepared_mesh_path = practical_dir / "prepared_mesh.npz"
mesh = prepare_course_mesh(config, prepared_mesh_path)
print({"nodes": mesh.n_nodes, "T3_elements": mesh.n_elems,
       "boundary_sets": sorted(mesh.node_sets)})

### Set the initial notch and boundary conditions

The Boolean mask selects centreline nodes up to the nominal notch endpoint. Hold them at $d=1$. The boundary helper prescribes $u_y=+\delta/2$ at the top, $u_y=-\delta/2$ at the bottom and $u_x=0$ on every exterior edge. Here $\delta$ denotes the total separation.

In [ ]:
from phast.physics.boundary_conditions import symmetric_tension_bcs
from phast.physics.material import Material
from phast.solvers.staggered_solver import StaggeredSolver
nodes = mesh.nodes
precrack = ((nodes[:, 1] - height / 2).abs() < 1.e-12)
precrack &= nodes[:, 0] <= notch_length + 1.e-12
bcs = symmetric_tension_bcs(mesh, disp=config["loading"]["total_symmetric_vertical_displacement"])
bcs.add_pf_dirichlet(torch.where(precrack)[0], value=1.0)

### Construct the material and solver

The material uses $E=1$, $\nu=0.3$, $G_c=0.0005$ and $\ell=0.15$. `StaggeredSolver` receives the mesh, material and boundary objects we just created. Seed its initial damage using the same notch mask.

In [ ]:
material = Material(**config["material"])
assert material.plane_stress is False
solver = StaggeredSolver(mesh, material, bcs, solver_configuration(config))
solver.d[precrack] = 1.0
assert solver.mesh is mesh and solver.bcs is bcs and solver.material is material
print({"locked_precrack_nodes": int(precrack.sum()), "kinematics": "plane strain"})

Locate the prescribed boundaries and the initial notch in these full-domain views. Expand the cell to inspect the plotting commands.

In [ ]:
nodes = mesh.nodes.numpy()
tri = mtri.Triangulation(nodes[:, 0], nodes[:, 1], mesh.elements.numpy())
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), layout="constrained")
axes[0].triplot(tri, color="#b3bbc3", lw=.16)
for name, color in (("top", "#245a81"), ("bottom", "#245a81"), ("left", "#087f83"), ("right", "#087f83")):
    boundary = mesh.node_sets[name].numpy()
    axes[0].scatter(nodes[boundary, 0], nodes[boundary, 1], s=5, c=color)
axes[0].scatter(nodes[precrack, 0], nodes[precrack, 1], s=16, c="#d46b27", label=r"Locked $d=1$ nodes")
axes[0].legend(loc="upper right")
axes[0].set_title("Actual T3 mesh and constrained nodes")
field = axes[1].tripcolor(tri, solver.d.numpy(), cmap="magma", vmin=0, vmax=1, shading="gouraud")
axes[1].set_title("Initial damage, before loading")
for ax in axes:
    ax.set(xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal")
    ax.grid(False)
fig.colorbar(field, ax=axes[1], label=r"Damage $d$: 0 intact; 1 damaged")
show_figure(fig, "tiny_notched_tension_initial_notch.png", "Actual imported triangular mesh with its constrained exterior nodes and locked centreline nodes; separate full initial damage field with scale zero to one.")
print({"locked_precrack_nodes": int(precrack.sum()), "last_locked_x": float(mesh.nodes[precrack, 0].max()),
       "material_kinematics": "plane stress" if material.plane_stress else "plane strain"})

### Apply the load and solve

The 60 load factors advance a sequence of quasistatic equilibria. At each increment, PhAST alternates mechanics, driving history and damage until the relative $L^2$ changes in both fields are below $10^{-5}$.

`solve_prepared_case` advances the existing `solver`. Its loop sets `bcs.load_factor`, calls `solver.step_full()`, records the fields, and checks displacement constraints, damage bounds, irreversibility and convergence at every increment. [Open that short workflow and its checks](https://github.com/CEMS-Lab/autumn-school/blob/main/notebooks/day2_helpers/forward_workflow.py).

In [ ]:
loading = config["loading"]
load_factors = torch.linspace(loading["first_load_factor"], loading["last_load_factor"], loading["n_load_steps"])
print({"increments": len(load_factors), "first_separation": float(load_factors[0] * loading["total_symmetric_vertical_displacement"]),
       "final_separation": float(load_factors[-1] * loading["total_symmetric_vertical_displacement"])})
result = solve_prepared_case(solver, config, load_factors, precrack, phast_info, retain_history=True)
assert result["solver"] is solver and solver.mesh is mesh
summary = result["metadata"]["summary"]
assert summary["incremental_damage_outside_sum"] > 1.0
assert summary["incremental_damage_outside_max"] > 1.e-3
assert summary["solve_seconds"] < config["limits"]["solver_seconds_hard"]
print({"solve_seconds": round(summary["solve_seconds"], 2), "load_steps": summary["load_steps"],
       "checks": result["metadata"]["checks"]})

### Save and reopen the fields

The original NPZ stores the named fields and four selected snapshots. A separate history NPZ stores the seeded state and every accepted displacement/damage state. JSON records the case, source revision, response trace and file hashes. Reopening checks file integrity and reproduces every array. These files support independent post-processing.

In [ ]:
result["metadata"]["source_hashes"] = {
    name: digest_file(COURSE_ROOT / name) for name in (
        "notebooks/day2_helpers/forward_workflow.py",
        "notebooks/day2_helpers/classroom_visuals.py",
        "configs/day2_forward/tiny_notched_tension.json")}
notebook_path = "notebooks/classroom/01_simulate_fracture.ipynb"
result["metadata"]["notebook_source"] = {
    "path": notebook_path,
    "cell_source_sha256": notebook_source_hash(COURSE_ROOT / notebook_path)}

In [ ]:
field_path, metadata_path = save_results(result, practical_dir)
saved, record = load_results(field_path, metadata_path)
for name, array in result["arrays"].items():
    np.testing.assert_array_equal(saved[name], array)
assert record["config"] == config
print({"fields_file": field_path.name, "case_file": metadata_path.name,
       "displacement_shape": saved["final_displacement"].shape})

## Interpret the computed fields

The vertical displacement and magnified boundary outline show the imposed motion. Initial, first-increment and final damage use a common colour scale. The additional increment plot isolates damage accumulated after the first equilibrium step. All views use the reopened arrays.

In [ ]:
nodes, triangles = saved["nodes"], saved["elements"]
tri = mtri.Triangulation(nodes[:, 0], nodes[:, 1], triangles)
u = saved["final_displacement"]
deformation_scale = 5
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), layout="constrained")
umax = np.abs(u[:, 1]).max()
image = axes[0].tripcolor(tri, u[:, 1], cmap="coolwarm", vmin=-umax, vmax=umax, shading="gouraud")
fig.colorbar(image, ax=axes[0], label=r"$u_y$ [dimensionless]")
axes[0].set_title("Vertical displacement at final load")
deformed = nodes + deformation_scale * u
for name in ("left", "right", "top", "bottom"):
    idx = saved["boundary_" + name]
    axes[1].plot(nodes[idx, 0], nodes[idx, 1], color="#9da6af", lw=1, linestyle="--")
    axes[1].plot(deformed[idx, 0], deformed[idx, 1], color="#245a81", lw=2)
axes[1].plot([], [], "--", color="#9da6af", label="Reference outline")
axes[1].plot([], [], color="#245a81", label=f"Deformed outline ({deformation_scale}×)")
axes[1].legend(loc="center")
axes[1].set_title("Symmetric separation of the boundaries")
for ax in axes:
    ax.set(xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal")
show_figure(fig, "tiny_notched_tension_displacement.png", "Final vertical displacement with a symmetric color range, alongside the reference and five-times magnified deformed specimen outlines.")

Compare damage across the complete specimen before loading, after the first increment and at the final separation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.5), layout="constrained")
for ax, name, title in zip(axes, ("seeded_damage", "first_damage", "final_damage"),
                         ("Initial condition", "First load increment", "Final load increment")):
    field = ax.tripcolor(tri, saved[name], cmap="magma", vmin=0, vmax=1, shading="gouraud")
    ax.set(title=title, xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal")
    ax.grid(False)
fig.colorbar(field, ax=axes, label=r"Damage $d$: 0 intact; 1 damaged")
show_figure(fig, "tiny_notched_tension_damage.png", "Full initial, first-increment and final PhAST damage fields, all using the same damage scale from zero to one.")

This increment separates the initial damaged centreline from subsequent diffuse evolution. The current example illustrates damage accumulation under the stated loading and restraints.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
field = ax.tripcolor(tri, saved["incremental_damage"], cmap="viridis", shading="gouraud", vmin=0)
ax.set(title="Damage accumulated after the first increment", xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal")
ax.grid(False)
fig.colorbar(field, ax=ax, label=r"$\max(d_{60}-d_1,0)$")
show_figure(fig, "tiny_notched_tension_increment.png", "Full spatial field of positive damage accrued after the first increment, with a labelled quantitative color scale.")

The top reaction sums the internal vertical forces at prescribed top nodes. Read this curve alongside the fields. The nodal damage sum depends on mesh density; the iteration count describes coupling effort.

In [ ]:
trace = record["trace"]
separation = np.array([row["applied_separation"] for row in trace])
reaction = np.array([row["reaction_top_y"] for row in trace])
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.5), layout="constrained")
axes[0].plot(separation, reaction, color="#245a81")
axes[0].set(ylabel=r"Top reaction $F_y$ [dimensionless]", title="Reaction–separation response")
axes[1].plot(separation, [row["damage_outside_sum"] for row in trace], color="#d46b27")
axes[1].set(ylabel="Nodal damage sum outside precrack", title="Diffuse damage evolution")
axes[2].plot(separation, [row["stagger_iterations"] for row in trace], color="#087f83")
axes[2].set(ylabel="Staggered iterations", title="Coupling effort per increment")
for ax in axes:
    ax.set_xlabel(r"Separation $\delta$ [dimensionless]")
show_figure(fig, "tiny_notched_tension_response.png", "Reaction against prescribed separation, nodal damage sum outside the initial notch, and staggered coupling iterations at each increment.")

### Follow the accepted states

The animation shows the seeded notch followed by all 60 accepted quasistatic increments, with damage fixed to the same 0–1 colour scale. Each frame corresponds to saved numerical arrays; the reaction marker identifies its load. Playback speed is a viewing choice. The static full-domain figures above remain available for comparison. This case shows diffuse damage accumulation under the stated loading.

In [ ]:
from IPython.display import HTML
from day2_helpers.forward_workflow import load_history
from day2_helpers.classroom_visuals import fracture_evolution, response_table
retained_history = load_history(metadata_path, saved, record)
animation_path = fracture_evolution(saved, record, practical_dir / "tiny_notched_tension_evolution.gif", history=retained_history)
display(Image(filename=str(animation_path), format="gif",
              alt="Seeded notch and 60 accepted PhAST damage states with their reaction history."))
response_csv, response_html = response_table(record, practical_dir)
display(HTML(response_html))
print({"animation": animation_path.name, "all_increment_results": response_csv.name,
       "original_fields": field_path.name})

### Compare a smaller separation

For a fixed damage field, halving displacement halves strain and reduces elastic energy to one quarter. The coupled damage response also changes. Copy the case, halve its maximum separation, and create a fresh solver on the same mesh.

In [ ]:
smaller_config = copy.deepcopy(config)
smaller_config["loading"]["total_symmetric_vertical_displacement"] *= .5
smaller_bcs = symmetric_tension_bcs(mesh, disp=smaller_config["loading"]["total_symmetric_vertical_displacement"])
smaller_bcs.add_pf_dirichlet(torch.where(precrack)[0], value=1.0)
smaller_material = Material(**smaller_config["material"])
smaller_solver = StaggeredSolver(mesh, smaller_material, smaller_bcs, solver_configuration(smaller_config))
smaller_precrack = precrack.clone()
smaller_solver.d[smaller_precrack] = 1.0

Execute the same load factors and reopen the second case. Both cases keep the same coordinates, triangles and material values.

In [ ]:
smaller_result = solve_prepared_case(smaller_solver, smaller_config, load_factors, smaller_precrack, phast_info, retain_history=True)
smaller_fields, smaller_json = save_results(smaller_result, practical_dir, "tiny_notched_tension_half_load")
smaller_saved, smaller_record = load_results(smaller_fields, smaller_json)
for name, array in smaller_result["arrays"].items():
    np.testing.assert_array_equal(smaller_saved[name], array)
assert np.array_equal(smaller_saved["nodes"], saved["nodes"])
assert np.array_equal(smaller_saved["elements"], saved["elements"])
outside = ~saved["precrack"]
assert smaller_saved["final_damage"][outside].sum() < saved["final_damage"][outside].sum()
assert smaller_record["trace"][-1]["applied_separation"] == .5 * record["trace"][-1]["applied_separation"]

Inspect both final fields on the same scale. The assertion checks the lower nodal damage sum for this particular smaller-load calculation; the full fields show where that difference occurs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6), layout="constrained")
for ax, data, title in zip(axes[:2], (smaller_saved, saved), (r"Final damage: $\delta=0.02$", r"Final damage: $\delta=0.04$")):
    field = ax.tripcolor(tri, data["final_damage"], cmap="magma", vmin=0, vmax=1, shading="gouraud")
    ax.set(title=title, xlabel=r"$x$ [dimensionless]", ylabel=r"$y$ [dimensionless]", aspect="equal")
    ax.grid(False)
fig.colorbar(field, ax=axes[:2], label=r"Damage $d$")
for case_record, style, label in ((record, "-", r"$\delta_{\max}=0.04$"), (smaller_record, "--", r"$\delta_{\max}=0.02$")):
    axes[2].plot([r["applied_separation"] for r in case_record["trace"]], [r["reaction_top_y"] for r in case_record["trace"]], style, label=label)
axes[2].set(title="Response for each loading history", xlabel=r"$\delta$ [dimensionless]", ylabel=r"$F_y$ [dimensionless]")
axes[2].legend()
show_figure(fig, "tiny_notched_tension_comparison.png", "Final damage for half and full imposed separation on a shared zero-to-one color scale, with both reaction–separation curves.")
print({"full_load_damage_sum": float(saved["final_damage"][outside].sum()),
       "half_load_damage_sum": float(smaller_saved["final_damage"][outside].sum()),
       "half_load_solve_seconds": round(smaller_record["summary"]["solve_seconds"], 2)})

The comparison table uses each run's final accepted increment. Separate history NPZ files include every accepted displacement and damage state; each CSV contains all 60 response rows at full precision. The nodal damage sum is a mesh-dependent descriptor.

In [ ]:
from day2_helpers.classroom_visuals import numeric_table_html, save_table
smaller_history = load_history(smaller_json, smaller_saved, smaller_record)
half_csv, _ = response_table(smaller_record, practical_dir, "tiny_notched_tension_half_load")
comparison_rows = [dict(case=label, **case["trace"][-1]) for label, case in
                   (("Full separation", record), ("Half separation", smaller_record))]
columns = [("case", "Case"), ("applied_separation", "Final separation"),
           ("reaction_top_y", "Top reaction"), ("damage_outside_sum", "Nodal damage sum outside notch")]
save_table(comparison_rows, practical_dir / "tiny_notched_tension_comparison.csv")
display(HTML(numeric_table_html(comparison_rows, columns, "Final states on the same mesh and scales")))

### Example outline · P1-R01 · A short propagating crack

**Learning question:** How does a resolved crack front advance as loading continues?

**Planned content.** Compare full-domain displacement and damage fields with the reaction history from a selected public PhAST propagation example.

### Animation storyboard · P1-A01 · Geometry to damage

**Current notebook visual.** The accepted-state GIF above shows the seeded notch followed by all 60 computed damage states with their response history. The geometry and mesh figures supply the preceding static views. A presenter-controlled composition can combine those stages using the retained outputs.

## Key takeaways

- The configuration drives geometry, mesh, material, constraints and loading through one connected object chain.
- The initial phase-field notch is a locked nodal damage condition whose endpoint follows the mesh spacing.
- Quasistatic staggered updates couple mechanical equilibrium with diffuse damage evolution; inspect both fields and reactions.
- Portable NPZ/JSON results preserve the numerical arrays and case information needed for independent post-processing.

## Student exercises

Work through both questions, then open the hints and worked answers.

### Exercise 1: Count the mesh and the locked precrack

The configuration uses a $4.0\times2.0$ rectangle, $n_x=128$ and $n_y=64$ rectangular cells, and splits each cell into two T3 triangles. The centreline precrack threshold has length $0.8$.

1. Derive the number of nodes and T3 elements.
2. Compute the horizontal node spacing, identify the last grid node satisfying $x\le0.8$, and count the locked centreline nodes.
3. Match your counts to the geometry/mesh cell in the notebook and explain the discretisation effect at the nominal notch endpoint.

<details><summary>Hint</summary>

A structured grid with $n_x$ cells has $n_x+1$ nodes in that direction. A T3 split contributes two triangles per rectangle. For the notch, first find $\Delta x=4/128$.

</details>

<details><summary>Worked Solution</summary>

1. The nodal grid has

$$
N=(128+1)(64+1)=129\times65=8385.
$$

There are $128\times64$ rectangles and two T3 elements per rectangle, so

$$
N_e=2(128)(64)=16384.
$$

2. The horizontal spacing is $\Delta x=4/128=0.03125$. The nominal threshold spans $0.8/0.03125=25.6$ spacings, placing the nominal endpoint $x=0.8$ between grid nodes. The largest admissible integer index is $\lfloor25.6\rfloor=25$, whose coordinate is $25(0.03125)=0.78125$. Nodes with indices $0,\ldots,25$ are locked, giving $25+1=26$ centreline precrack nodes.

3. The setup gives **8385 nodes / 16384 T3 elements** and **precrack_nodes: 26**, so the hand count agrees. This is a discretisation effect: the configuration specifies a threshold at $0.8$, while the nodal Dirichlet mask actually ends at the last represented centreline node within that threshold, $x=0.78125$. The prepared tensor-mesh file stores these generated coordinates, connectivity and node labels.

</details>

### Exercise 2: Predict and test the effect of a smaller separation

The final notebook experiment changes only the total imposed separation from $0.04$ to $0.02$.

1. With damage held fixed, derive the factor by which strain and elastic energy change when the prescribed displacement is halved.
2. Use the saved files for the two calculations to compare their final damage fields on the same colour scale and their reaction–separation curves. Explain how the coupled damage update affects the response.
3. Identify the arrays and metadata another student needs to recreate these plots.

<details><summary>Hint</summary>

For fixed stiffness, displacement and strain scale linearly with the prescribed separation. Elastic energy is quadratic in strain. The coupled calculation updates damage as the loading progresses.

</details>

<details><summary>Worked Solution</summary>

1. Let $\boldsymbol\varepsilon$ be the small-strain tensor and $\mathbb C$ the elastic stiffness. Holding damage fixed gives $\boldsymbol\varepsilon_{1/2}=\boldsymbol\varepsilon/2$, so $\psi_{1/2}=\tfrac12(\boldsymbol\varepsilon/2):\mathbb C:(\boldsymbol\varepsilon/2)=\psi/4$.

2. The notebook executes both complete loading histories using the same imported mesh and material. It verifies that the smaller-separation calculation has a smaller final nodal damage sum outside the locked precrack. Compare the full fields using $0\le d\le1$ for both plots. As damage changes the stiffness, the coupled reaction curves reflect both the imposed displacement and damage evolution. A reaction ratio of exactly one half applies to a fixed-damage linear-elastic comparison.

3. The NPZ files supply coordinates, triangle connectivity, boundary node labels and final displacement/damage arrays. The JSON files supply the configuration, dimensionless conventions, public source revision and the per-step `applied_separation` and `reaction_top_y` values. The saved hashes and reload checks establish file integrity and array equality. These result files support portable post-processing; a simulation restart also requires internal history and solver state.

</details>

## Continue studying

[Detailed notebook 01](https://github.com/CEMS-Lab/autumn-school/blob/main/notebooks/01_phast_tiny_evolving_fracture.ipynb)

The complete source, attribution and bundled licences remain in the [course repository](https://github.com/CEMS-Lab/autumn-school/blob/main/ATTRIBUTION.md).

### Session timing

Elapsed session time includes reading and pauses between cells. The course runner measures uninterrupted computation separately from installation.

In [ ]:
session_seconds = time.perf_counter() - session_started
print({"setup_seconds": setup_summary["setup_seconds"],
       "elapsed_session_seconds": round(session_seconds, 2),
       "environment": setup_summary["environment"], "torch": torch.__version__})